# COSC-651 Week 3: Tool-Using Agents and Controlled Execution

Supports course learning outcomes 3, 5, and 7.

This week LedgerLine Assist decides which evidence to inspect next, rather than having code read every policy for every case. You will measure what that change gains and which access limits still hold.

Build the bounded agent version of Assist. The AI can propose an investigation step after seeing a result; the application validates and runs permitted requests. Test dependent calls, invalid proposals and stopping behavior, then compare the agent with the fixed workflow. Keep the customer's report separate from a verified importer defect. Your output remains a recommendation for support, not an executed software repair.

Read the weekly lab in Canvas for the assignment and case context.

## Experimental question

Can a bounded agent gather the evidence it needs while respecting the same task and execution limits as the fixed workflow?

Read the weekly lab and rubric before changing the starter. Fixture mode is a deterministic test double. Its results do not measure LLM capability.

## Environment and data

Use the installed course environment. Run the notebook from a clean kernel.

In [ ]:
from pathlib import Path
import json, sys
REPO_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                  if (p / 'src/maintenance_course/data.py').is_file()), None)
assert REPO_ROOT is not None, 'Open this notebook from a COSC-651-Labs checkout.'
sys.path[:0] = [str(REPO_ROOT / 'src'), str(REPO_ROOT)]
from maintenance_course.data import LAB_ROOT, DATA, cases, case_by_id
from maintenance_course.experiments import run_case, save_runs, payload, gather
from maintenance_course.baselines import direct, fixed
from maintenance_course.models import FixtureModel
from maintenance_course.tools import ToolRegistry
sys.path.insert(0, str(LAB_ROOT))
MODE = "fixture"  # Change only after configuring the course-approved live access.
print({"mode": MODE, "lab_root": str(LAB_ROOT), "development_cases": len(cases())})

## Provided baseline

Run and inspect this example before implementing your own functions. It demonstrates the supplied infrastructure.

In [ ]:
registry = ToolRegistry(case_by_id("D01"))
for name, args in [("diagnose_csv", {"fixture_id":"missing-date"}), ("read_document", {"doc_id":"dates-v2"}), ("diagnose_csv", {"fixture_id":"valid"}), ("approve", {})]:
    print(name, registry.call(name, args))
print("These are dispatcher checks; implement the model-directed loop separately.")

## Your implementation

Implement `agent(model, registry, case, max_steps=5)` in `student_work/week03.py`. Accept only a tool proposal or a final report, validate final evidence against `registry.observed`, and reject budgets above ten.

Set `RUN_STUDENT` to `True` after completing the named functions. The default leaves unfinished functions unexecuted; it does not mark the assignment complete.

In [ ]:
import student_work.week03 as student
RUN_STUDENT = False
if RUN_STUDENT:
    result = run_case(student.agent, case_by_id("D01"), mode=MODE)
    print(result)
else:
    print("Student implementation has not been evaluated. Complete the lab, then enable this cell.")

## Required experiment

1. **Inspect the tool surface.** Read `contracts.py`, `tools.py`, and `worker.py`. Make a table of tool name, exact arguments, permitted scope, success result, and failure result. Identify why the calculator is a restricted language rather than an arbitrary Python sandbox.
2. **Implement the loop.** At each decision, call the model with current observations. Dispatch permitted calls through the registry and append the result. Validate the final report outside the model. Count failed proposals and the final response against the decision budget.
3. **Demonstrate dependent calls.** Run D01 and another case requiring a different policy. Show the diagnostic result followed by the selected document read. Use at least two distinct tools in the successful trace and explain how an observation affected the next action.
4. **Exercise rejection paths.** Test an unknown tool, an extra argument, a wrong fixture identifier, a stale document, and budget exhaustion. Call the dispatcher directly as well as through an agent so a model's reluctance to request an invalid action cannot hide a missing code boundary.
5. **Test unavailable and hostile inputs.** Use D14 for denied access, D15 for timeout, and D11 for embedded instructions. Treat these as targeted boundary tests this week; expand the full development evaluation in Week 5. Confirm that no tool path can grant approval.
6. **Compare behavior and work.** Run the agent and fixed baseline on D01 through D10, with fixture and available live evidence separated. Report task success, observed tool sequence, calls, latency, and any boundary failure. Explain one resource or reliability tradeoff.

## Results and analysis

Add your executed experiment cells and case-level table here. State the case set, variant, mode, model, success dimensions, resource measurements, and exclusions. Trace a failure to its cause and explain the resulting architecture decision. Export selected evidence into the submission folder.

## Research, interaction, and disclosure

Link the primary-source research note, discussion and peer response, and AI-use disclosure. Check every required artifact against the weekly rubric.